# Лабораторна робота №5

**Створення інтерактивного блокноту Jupyter за допомогою Google Colab і Binder**

Виконав: Козюк Дмитро, група КН-123а  
Копія для запуску: **Google Colab**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zepif/big-data-lab5/blob/main/colab_notebook.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/zepif/big-data-lab5/main?labpath=binder_notebook.ipynb)

Репозиторій: [https://github.com/zepif/khpi_big_data_lab_5](https://github.com/zepif/khpi_big_data_lab_5)


Перед виконанням прикладу встановлюємо потрібні пакети та завантажуємо файл послідовності у тимчасове середовище.

In [ ]:
%pip install -q biopython folium matplotlib

from pathlib import Path
from urllib.request import urlretrieve

data_file = Path("NC_005816.gb")
if not data_file.exists():
    urlretrieve("https://raw.githubusercontent.com/zepif/khpi_big_data_lab_5/main/NC_005816.gb", data_file)

print("Середовище: Google Colab")
print("Дані готові:", data_file, data_file.stat().st_size, "байтів")


This simple notebook demonstrates how users can interleave text, code, and results in a single document. We start with a simple calculation -- computing the first 25 numbers in the Fibonacci sequence, where each value equals the sum of the two previous values. The Jupyter notebook allows us to express that mathematically, using the typesetting language $\LaTeX{}$: $$F_n = F_{n-1} + F_{n-2}$$
Thus, the sequence is: 0, 1, 1, 2, 3, 5, 8, ...

The first cell contains an IPython 'magic' code, '%matplotlib', which allows the notebook to display plots inline, in the body of the notebook.

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# calculate the first 25 Fibonacci numbers
f1 = 0
f2 = 1
ar = [f1, f2] # a list to hold the computed values. We know the first two numbers

# we only need to run our calculation 23 times, because positions 1 and 2 are known
for i in range (23):
    f3 = f1 + f2
    ar.append (f3)
    f1 = f2
    f2 = f3
    
print (ar) # below, you see the output of the code itself.

Plot the data

In [ ]:
fig, ax = plt.subplots()
ax.plot (range(25), ar, "ro")
## uncomment the following call to ax.plot() (by removing the leading '#') and select 
## 'Cell > Run All' (in Binder) or 'Run > Run All Cells' (Jupyter) to change the graph below
# ax.plot (range(25), ar)
ax.set (xlabel = "Sequence No.", ylabel = "Fibonacci No.", 
       title = "The First 25 Fibonacci Numbers")
plt.show()

Anything you can do programmatically, can be documented in a notebook. Here we'll do some simple sequence analysis with Biopython. The following example is adapted from the [Biopython tutorial](http://biopython.org/DIST/docs/tutorial/Tutorial.html). 

First, we'll read in a Genbank-formatted file, which represents a circular DNA, called a plasmid, from the bacterium, *Yersinia pestis*. 

In [ ]:
from Bio import SeqIO

record = SeqIO.read(data_file, "genbank")
record.seq


How long is the DNA, and how which genes does it encode?

In [ ]:
print("Length:", len(record.seq), "bp")
print("No. features:", len(record.features),"\n")

for i,feat in enumerate (record.features):
    if (feat.type == "CDS"):
        product = feat.qualifiers['product'][0]
        print ("Feat. %d: %s" % (i, product))

This plasmid encodes 41 features, including 10 genes. Let's focus on feature #18, which encodes a 'hypothetical protein'.

In [ ]:
feat = record.features[18]
print ("Gene location\nStart:", feat.location.start, "\nEnd:", feat.location.end)
print ("GeneID:", feat.qualifiers.get("db_xref"))

These data show us that the gene is located between bases 3485 and 3857. Let's retrieve that segment and translate it.

In [ ]:
my_gene = record[3485:3857]
print (my_gene.seq.translate(table="Bacterial"))

Now, let's find out more about this gene, using the NCBI Entrez database. We learned the accession number above: `GeneID: ['GI:45478715', 'GeneID:2767720']`.

**20 May 2020**: Updated the following two cells to reflect changes in NCBI's Entrez database. 

In [ ]:
from Bio import Entrez

In [ ]:
Entrez.email = "dmytrokoziuk68@gmail.com"
with Entrez.efetch(db="protein", rettype="gb", retmode="text", id="AAS58761") as handle:
    print(handle.read())


Jupyter also supports interactive data exploration. Here we'll add an interactive map, just as in our June [mapping feature](https://www.nature.com/articles/d41586-018-05331-6) -- something you might do when working with geospatial data. This requires the Python Leaflet library, folium. 

In [ ]:
import folium

Now we create a simple map: a few points in London, Oxford and Cambridge, overlaid on either a street map, or on a map of geological data provided by the [Macrostrat Project](https://macrostrat.org/). 

In [ ]:
coords = { 
    0: { "name": "Nature", "lat": 51.533925, "long": -0.121553 },
    1: { "name": "Francis Crick Institite", "lat": 51.531877, "long": -0.128767 },
    2: { "name": "MRC Laboratory for Molecular Cell Biology", "lat": 51.524435, "long": -0.132495 },
    3: { "name": "Kings College London", "lat": 51.511573, "long": -0.116083 },
    4: { "name": "Imperial College London", "lat": 51.498780, "long": -0.174888 },
    5: { "name": "Cambridge University", "lat": 52.206960, "long": 0.115034 },
    6: { "name": "Oxford University", "lat": 51.754843, "long": -1.254302 },
    7: { "name": "Platform 9 3/4", "lat": 51.532349, "long": -0.123806 }
}

In [ ]:
m = folium.Map(location = [51.8561, -0.2966], tiles = 'CartoDB positron', zoom_start = 9)

# add the locations to the map
for key in coords.keys():
    folium.CircleMarker(
    location=[coords[key]['lat'], coords[key]['long']],
    popup=coords[key]['name'],
    color=('crimson' if coords[key]['name'] == 'Nature' else 'blue'),
    fill=False,
    ).add_to(m)

# pull in the Macrostrat tile layer
folium.TileLayer(tiles='https://tiles.macrostrat.org/carto/{z}/{x}/{y}.png', 
                 attr='Macrostrat', name='Macrostrat').add_to(m)
folium.LayerControl().add_to(m) # allow user to switch between layers
folium.LatLngPopup().add_to(m) # click on the map to get Lat/Long in a popup

Draw the map. **Note that this map is interactive**: you can zoom, pan, click the points of interest, and alternate between the two layers (by clicking on the tiles icon in the upper-right corner of the map). If you click anywhere on the map, a popup will appear showing the latitude and longitude of that position.

In [ ]:
m

Document our session, for [computational reproducibility](https://www.nature.com/articles/d41586-018-05990-5)!

In [ ]:
import sys
import IPython

environment = "Google Colab" if "google.colab" in sys.modules else "Binder/Jupyter"
print("Середовище:", environment)
print(IPython.sys_info())


In [ ]:
!pip freeze | grep -E 'folium|matplotlib|biopython'